# 딥러닝 음성 잡음 제거 · 추론 노트북

`input/`의 모든 noisy WAV를 학습된 가중치로 잡음 제거해 `output/`에 **같은 파일명**으로 저장합니다. 학습은 하지 않습니다.

**실행 방법:** `input/`에 16 kHz 모노 WAV를 넣고 위에서부터 모든 셀을 실행하세요.

- 이 노트북은 저장소 최상위 폴더(`model.py`, `inference.py`, `weights/`가 있는 곳)에 있어야 합니다.
- 모델: DPCRN 3개 + BiLSTM 2개 가중 앙상블 + 시간 이동 TTA (검증 평균 PESQ-WB 2.8269)
- 출력: 16 kHz 모노, 입력과 같은 표본 수

In [ ]:
# 1. 필요한 패키지 중 없는 것만 설치합니다.
import importlib.util
import subprocess
import sys

packages = {"torch": "torch>=2.1", "numpy": "numpy", "soundfile": "soundfile"}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
print("패키지 준비 완료")

In [ ]:
# 2. 경로와 설정
from pathlib import Path

import numpy as np
import soundfile as sf
import torch


def notebook_dir():
    # VS Code는 노트북 경로를 __vsc_ipynb_file__에 넣어 주고, Jupyter는 노트북 폴더에서 시작합니다.
    vscode_path = globals().get("__vsc_ipynb_file__")
    return Path(vscode_path).resolve().parent if vscode_path else Path.cwd().resolve()


ROOT = notebook_dir()
if not (ROOT / "inference.py").is_file():
    raise FileNotFoundError(f"{ROOT}에 inference.py가 없습니다. 노트북을 저장소 최상위 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT))
from inference import TTA_SHIFTS, enhance, load_ensemble   # 모델 정의는 model.py
from model import SAMPLE_RATE

INPUT_DIR = ROOT / "input"
OUTPUT_DIR = ROOT / "output"
CONFIG_PATH = ROOT / "weights" / "ensemble.json"
USE_TTA = True        # False로 바꾸면 약 4배 빠르지만 검증 PESQ가 약 0.0075 낮습니다.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shifts = TTA_SHIFTS if USE_TTA else (0,)
print(f"저장소 {ROOT}")
print(f"장치 {device} | TTA {'켬' if USE_TTA else '끔'}")

In [ ]:
# 3. 앙상블 가중치 불러오기
members = load_ensemble(CONFIG_PATH, device)
for name, model, weight in members:
    parameters = sum(p.numel() for p in model.parameters())
    print(f"{name:12s} {type(model).__name__:11s} 비중 {weight:.2f} | 파라미터 {parameters:,}")

In [ ]:
# 4. input/ → output/ 잡음 제거
wav_paths = sorted(p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() == ".wav")
if not wav_paths:
    raise FileNotFoundError(f"{INPUT_DIR}에 WAV 파일이 없습니다.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for index, in_path in enumerate(wav_paths, 1):
    info = sf.info(in_path)
    if info.samplerate != SAMPLE_RATE or info.channels != 1:
        raise ValueError(f"16 kHz 모노 WAV가 필요합니다: {in_path.name} "
                         f"({info.samplerate} Hz, {info.channels}채널)")
    noisy, _ = sf.read(in_path, dtype="float32")
    enhanced = enhance(members, noisy, device, shifts)
    assert enhanced.shape == noisy.shape            # 입력과 같은 표본 수
    if info.subtype.startswith("PCM"):
        enhanced = np.clip(enhanced, -1.0, 1.0)
    sf.write(OUTPUT_DIR / in_path.name, enhanced, SAMPLE_RATE, subtype=info.subtype)
    print(f"[{index}/{len(wav_paths)}] {in_path.name}")

extra = ({p.name for p in OUTPUT_DIR.iterdir() if p.suffix.lower() == ".wav"}
         - {p.name for p in wav_paths})
if extra:
    print(f"주의: output/에 입력에 없는 WAV {len(extra)}개가 있습니다. 제출 전에 지우세요: {sorted(extra)[:5]}")
print(f"완료: {len(wav_paths)}개 파일을 {OUTPUT_DIR}에 저장했습니다.")